### Corresponds to ```/home/cir/tdeimel/autoscoRA/autoscoRA_Pipeline/evaluation/figures/interrater_reliability.py```

Q: Why is ICC different compared to calculation with pinguines?



In [ ]:

# from scoring.src.io_scoring_method import mandatory_train_val_ids
# import input.constants.augmentation_constants as augm
# import patch_extraction.io_patch_extraction as iop
# import patch_extraction.patch_extraction_func as pe
import os
import pandas as pd
import numpy as np
from copy import copy, deepcopy
import re
from sklearn.metrics import cohen_kappa_score
from rpy2.robjects import DataFrame, FloatVector, IntVector
from rpy2.robjects.packages import importr
r_icc = importr("ICC")
r_irr = importr("irr")
r_iccp = importr("psych")
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import yaml
from typing import Literal

READ_PATHS = "YAML" # "CONST.py", "HARDCODED"


def mandatory_train_val_ids(segm_to_train=True, double_to_val=True, missing_to_train=False, conflict='train',
                            get_paths: Literal["YAML", "CONST.py", "HARDCODED"] = "YAML",
                            input_constants_yml_path = "/home/cwatzenboeck/code/RA/ra_utils/ra_utils/autoscora/autoscorRA_Pipeline/input/constants/input_constants_cw.yml"):
    if get_paths == "CONST.py":
        import input.constants.input_constants as const
        print("Loading paths from: ", "'const.py'")
        H_score_path = const.PAT_DF_MANUAL_PATH
        F_score_path = const.F_PAT_DF_MANUAL_PATH
        H_segm_path = const.JOINTS_PATH_GT_100
        F_segm_path = const.F_JOINTS_PATH_GT_100
        double_score_path = const.H_F_DOUBLE_SCORE_PATH
        pass

    elif get_paths == "YAML":
        input_constants_yml_path = "/home/cwatzenboeck/code/RA/ra_utils/ra_utils/autoscora/autoscorRA_Pipeline/input/constants/input_constants_cw.yml"        
        print("Loading paths from: ", input_constants_yml_path)
        with open(input_constants_yml_path, "r") as f:
            const = yaml.safe_load(f)
        H_score_path = const["PAT_DF_MANUAL_PATH"]
        F_score_path = const["F_PAT_DF_MANUAL_PATH"]
        H_segm_path = const["JOINTS_PATH_GT_100"]
        F_segm_path = const["F_JOINTS_PATH_GT_100"]



        double_score_path = const["H_F_DOUBLE_SCORE_PATH"]


    elif get_paths == "HARDCODED": 
            H_score_path = \
                '/home/cir/tdeimel/autoscoRA/autoscoRA_Preprocessing/output/data_split/' \
                'pat_df_medstream_manual_H_dp_img_of_int_2_' \
                'summary_cols_segm_sets_RL_stratmean_split6543_chosen345_2019-05-03_21-12-37.csv'
            F_score_path = \
                '/home/cir/tdeimel/autoscoRA/autoscoRA_Preprocessing/output/pat_df_manual_FEET/data_split/' \
                'pat_df_medstream_manual_F_dp_img_of_int_2_' \
                'summary_cols_segm_sets_RL_stratmean_split6543_chosen345_2021-05-25_13-45-01_fixedComTBD.csv'

            H_segm_path = "/home/cir/tdeimel/autoscoRA/autoscoRA_Annotation/output/annotations/" \
                        "hands_combined/all_hand_joints_100_onlynecessaryjoints.csv"

            F_segm_path = "/home/cir/tdeimel/autoscoRA/autoscoRA_Annotation/output/annotations/" \
                        "feet/all_foot_joints_100.csv"

            double_score_path = "/home/cir/tdeimel/autoscoRA/autoscoRA_Preprocessing/output/double_scoring/" \
                                "Re_Scoring_50_v2_FINAL_pseudonymized.csv"
    else:
        raise NotImplementedError

    H = pd.read_csv(H_score_path)
    F = pd.read_csv(F_score_path)
    H_segm = pd.read_csv(H_segm_path)
    F_segm = pd.read_csv(F_segm_path)
    D = pd.read_csv(double_score_path)

    mand_train = []
    mand_val = []

    if missing_to_train:
        # missing extremities = MUST BE IN train set
        H_miss = (H
                  .groupby('id_RO_nr')[['laterality_manual']]
                  .agg(lambda x: x.sum() in ['RL', 'LR'])
                  .query("laterality_manual == False")
                  .index
                  .tolist()
                  )
        # a = H[H.id_nr==203].sort_values(by="RO_datum")
        F_miss = (F
                  .groupby('id_RO_nr')[['laterality_manual', 'bodypart_manual']]
                  .agg(
            lambda x: (x['laterality_manual'].sum() in ['RL', 'LR']) and (x['bodypart_manual'].sum() in ['HF', 'FH']))
                  .query("laterality_manual == False")
                  .index
                  .tolist()
                  )
        FH_miss = list(set([f for f in F.id_RO_nr if f not in list(H.id_RO_nr)] + [h for h in H.id_RO_nr if
                                                                                   h not in list(F.id_RO_nr)]))
        # F.query("id_nr == 689").filter(['id_RO_nr', 'laterality_manual', 'bodypart_manual'])
        # H.query("id_nr == 689").filter(['id_RO_nr', 'laterality_manual', 'bodypart_manual'])

        mand_train = mand_train + list(set([re.sub("_.*", "", i) for i in H_miss] +
                                           [re.sub("_.*", "", i) for i in F_miss] +
                                           [re.sub("_.*", "", i) for i in FH_miss]))
    if segm_to_train:
        # segm annotation pat = MUST BE IN train set
        mand_train = mand_train + list(
            set([re.sub("_.*", "", i) for i in H_segm.img] + [re.sub("_.*", "", i) for i in F_segm.img]))

    if double_to_val:
        # double scoring available = MUST BE IN final validation set
        mand_val = mand_val + list(set([str(int(i)) for i in D.id_nr if not np.isnan(i)]))

    if conflict == 'train':
        # if ids in "MUST BE TRAIN" and "MUST BE VAL", put them in train
        mand_val_return = [i for i in mand_val if i not in mand_train]
        excluded_mand_val = [i for i in mand_val if i in mand_train]
        # [i in [re.sub("_.*", "", i) for i in (H_segm.filter(['img']))['img']] for i in excluded_mand_val]
        # [i in [re.sub("_.*", "", i) for i in (F_segm.filter(['img']))['img']] for i in excluded_mand_val]

        mand_train_return = list(set(mand_train + excluded_mand_val))
        # print("mand_train NOT in excluded_mand_val")
        # print([i for i in mand_train if i not in excluded_mand_val])
        # print(list(set(mand_train)) == mand_train_return)

    elif conflict == 'val':
        raise NotImplementedError()
    else:
        raise ValueError()

    # von 100 segm img in F (bzw. bissl anderen id_RO_nr (aber gleichen id_nr) 100 in H) sind nur 80 unique id_nr
    return {"train": mand_train_return, "val": mand_val_return}


In [ ]:


wrist_type = "single"  # ['sum', 'single', 'individual_presum']
display_agreement = "icc"  # ['icc', 'kappa']
icc_type = 'ICC3'  # ['ICC1', 'ICC2', 'ICC3', 'ICC1k', 'ICC2k', 'ICC3k']
# wrist_type = "individual_presum"  # doesn't work because don't have an entry for "probs" in loaded results csv
# wrist_type = "single"


# files

if READ_PATHS: 
    if READ_PATHS == "CONST.py":
        import input.constants.input_constants as const
        print("Loading paths from: ", "'const.py'")
        H_score_path = const.PAT_DF_MANUAL_PATH
        F_score_path = const.F_PAT_DF_MANUAL_PATH
        double_score_path = const.H_F_DOUBLE_SCORE_PATH
        pass

    elif READ_PATHS == "YAML":
        input_constants_yml_path = "/home/cwatzenboeck/code/RA/ra_utils/ra_utils/autoscora/autoscorRA_Pipeline/input/constants/input_constants_cw.yml"        
        print("Loading paths from: ", input_constants_yml_path)
        with open(input_constants_yml_path, "r") as f:
            const = yaml.safe_load(f)
        H_score_path = const["PAT_DF_MANUAL_PATH"]
        F_score_path = const["F_PAT_DF_MANUAL_PATH"]
        double_score_path = const["H_F_DOUBLE_SCORE_PATH"]


    elif READ_PATHS == "HARDCODED": 
            H_score_path = \
                '/home/cir/tdeimel/autoscoRA/autoscoRA_Preprocessing/output/data_split/' \
                'pat_df_medstream_manual_H_dp_img_of_int_2_' \
                'summary_cols_segm_sets_RL_stratmean_split6543_chosen345_2019-05-03_21-12-37.csv'
            F_score_path = \
                '/home/cir/tdeimel/autoscoRA/autoscoRA_Preprocessing/output/pat_df_manual_FEET/data_split/' \
                'pat_df_medstream_manual_F_dp_img_of_int_2_' \
                'summary_cols_segm_sets_RL_stratmean_split6543_chosen345_2021-05-25_13-45-01_fixedComTBD.csv'
            double_score_path = "/home/cir/tdeimel/autoscoRA/autoscoRA_Preprocessing/output/double_scoring/" \
                                "Re_Scoring_50_v2_FINAL_pseudonymized.csv"
    else:
        raise NotImplementedError
    H_GS = pd.read_csv(H_score_path)
    F_GS = pd.read_csv(F_score_path)
    H_F_TD = pd.read_csv(double_score_path)



H_GS = H_GS.rename(columns=lambda s: re.sub("r_", "", s) if s.startswith('r_') else s)
F_GS = F_GS.rename(columns=lambda s: re.sub("r_", "", s) if s.startswith('r_') else s)
# H_F_TD = H_F_TD[[not i for i in (np.isnan(H_F_TD.id_nr))]]
H_F_TD = H_F_TD[[i == "WAHR" for i in H_F_TD.RO_datum_img_avail]]
H_F_TD = H_F_TD.astype({"id_nr": int})

# filter
mand_val = (mandatory_train_val_ids(segm_to_train=True, double_to_val=True, missing_to_train=False, conflict='train',
                                    get_paths="YAML"))['val']
H_GS = H_GS[[str(i) in mand_val for i in H_GS['id_nr']]]
F_GS = F_GS[[str(i) in mand_val for i in F_GS['id_nr']]]
H_F_TD = H_F_TD[[str(i) in mand_val for i in H_F_TD['id_nr']]]

print(len(list(set([str(i) for i in H_GS['id_nr'] if str(i) in mand_val]))))
print(len(list(set([str(i) for i in F_GS['id_nr'] if str(i) in mand_val]))))
print(len(list(set([str(i) for i in H_F_TD['id_nr'] if str(i) in mand_val]))))
print(len(list(set((H_GS[[str(i) in mand_val for i in H_GS['id_nr']]]).id_nr))))
print(len(list(set((F_GS[[str(i) in mand_val for i in F_GS['id_nr']]]).id_nr))))
print(len(list(set((H_F_TD[[str(i) in mand_val for i in H_F_TD['id_nr']]]).id_nr))))

# filter for those id_dates that occur in double scoring
H_GS['id_date'] = [re.search("[0-9]*_[0-9]*", i)[0] for i in H_GS['img_id']]
F_GS['id_date'] = [re.search("[0-9]*_[0-9]*", i)[0] for i in F_GS['img_id']]
H_F_TD['id_date'] = [str(row['id_nr']) + "_" + str(row['RO_datum']) for i, row in H_F_TD.iterrows()]

H_GS = H_GS[[i in list(H_F_TD['id_date']) for i in H_GS['id_date']]]
F_GS = F_GS[[i in list(H_F_TD['id_date']) for i in F_GS['id_date']]]
H_F_TD = H_F_TD[[i in list(H_GS['id_date']) or i in list(F_GS['id_date']) for i in H_F_TD['id_date']]]

print(len(list(set((H_GS[[str(i) in mand_val for i in H_GS['id_nr']]]).id_nr))))
print(len(list(set((F_GS[[str(i) in mand_val for i in F_GS['id_nr']]]).id_nr))))
print(len(list(set((H_F_TD[[str(i) in mand_val for i in H_F_TD['id_nr']]]).id_nr))))


# score_type --> score_name --> roi
# score_type --> score_name --> roi
scor_roi_matching_dict = {'JSN': {'CMCIII': ['SCD3'],
                                  'CMCIV': ['SCD4'],
                                  'CMCV': ['SCD5'],
                                  'MCPIII': ['SMD3'],
                                  'MCPII': ['SMD2'],
                                  'MCPI': ['SMD1'],
                                  'MCPIV': ['SMD4'],
                                  'MCPV': ['SMD5'],
                                  'PIPIII': ['SPD3'],
                                  'PIPII': ['SPD2'],
                                  'PIPIV': ['SPD4'],
                                  'PIPV': ['SPD5'],
                                  'Rad_Carp': ['SWR'],
                                  'Sca_Cap': ['SWR'],
                                  'Tra_Sca': ['SWR'],

                                  'MTPI': ['STD1'],
                                  'MTPII': ['STD2'],
                                  'MTPIII': ['STD3'],
                                  'MTPIV': ['STD4'],
                                  'MTPV': ['STD5'],
                                  'IP': ['SID1']
                                  },
                          'ERO': {'Base_MCIE': ['SCD1'],
                                  'IPIED': ['SPD1'],
                                  'IPIEP': ['SPP1'],
                                  'LunatE': ['SWR'],
                                  'MCPIIIED': ['SMD3'],
                                  'MCPIIIEP': ['SMP3'],
                                  'MCPIIED': ['SMD2'],
                                  'MCPIIEP': ['SMP2'],
                                  'MCPIED': ['SMD1'],
                                  'MCPIEP': ['SMP1'],
                                  'MCPIVED': ['SMD4'],
                                  'MCPIVEP': ['SMP4'],
                                  'MCPVED': ['SMD5'],
                                  'MCPVEP': ['SMP5'],
                                  'PIPIIIED': ['SPD3'],
                                  'PIPIIIEP': ['SPP3'],
                                  'PIPIIED': ['SPD2'],
                                  'PIPIIEP': ['SPP2'],
                                  'PIPIVED': ['SPD4'],
                                  'PIPIVEP': ['SPP4'],
                                  'PIPVED': ['SPD5'],
                                  'PIPVEP': ['SPP5'],
                                  'RadiusE': ['SWR'],
                                  'ScaphE': ['SWR'],
                                  'TrapE': ['SWR'],
                                  'UlnaE': ['SWR'],

                                  'MTPIEP': ['STP1'],
                                  'MTPIED': ['STD1'],
                                  'MTPIIEP': ['STP2'],
                                  'MTPIIED': ['STD2'],
                                  'MTPIIIEP': ['STP3'],
                                  'MTPIIIED': ['STD3'],
                                  'MTPIVEP': ['STP4'],
                                  'MTPIVED': ['STD4'],
                                  'MTPVEP': ['STP5'],
                                  'MTPVED': ['STD5'],
                                  'IPEP': ['SIP1'],
                                  'IPED': ['SID1']
                                  }
                          }

# roi --> score_type --> score_name
roi_scor_matching_dict = {roi: {'JSN': [], 'ERO': []}
                          for roi in list(set([roi_name for score_type, v0 in scor_roi_matching_dict.items()
                                               for score_name, v1 in v0.items()
                                               for roi_name in v1]))}

for roi_i, roi_dict_i in roi_scor_matching_dict.items():
    for score_type in roi_dict_i.keys():
        roi_scor_matching_dict[roi_i][score_type] = \
            [joint for joint, joint_rois in scor_roi_matching_dict[score_type].items()
             if roi_i in joint_rois]

ero_sum_h = ['MCPIE', 'MCPIIE', 'MCPIIIE', 'MCPIVE', 'MCPVE', 'IPIE', 'PIPIIE', 'PIPIIIE', 'PIPIVE', 'PIPVE']
ero_sum_f = ['MTPIE', 'MTPIIE', 'MTPIIIE', 'MTPIVE', 'MTPVE', 'IPE']
ero_pd_h = [i + "P" for i in ero_sum_h] + [i + "D" for i in ero_sum_h]
ero_pd_f = [i + "P" for i in ero_sum_f] + [i + "D" for i in ero_sum_f]
ero_finger = ['Base_MCIE', 'MCPIE', 'MCPIIE', 'MCPIIIE', 'MCPIVE', 'MCPVE', 'IPIE', 'PIPIIE', 'PIPIIIE', 'PIPIVE', 'PIPVE']
jsn_finger = ['CMCIII', 'CMCIV', 'CMCV', 'MCPI', 'MCPII', 'MCPIII', 'MCPIV', 'MCPV', 'PIPII', 'PIPIII', 'PIPIV', 'PIPV']
ero_wrist = ["RadiusE", "UlnaE", "ScaphE", "LunatE", "TrapE"]
jsn_wrist = ["Rad_Carp", "Sca_Cap", "Tra_Sca"]
ero_feet = ['MTPIE', 'MTPIIE', 'MTPIIIE', 'MTPIVE', 'MTPVE', 'IPE']
jsn_feet = ['MTPI', 'MTPII', 'MTPIII', 'MTPIV', 'MTPV', 'IP']


# function analogous to dplyr select(everthing())
def everything_after(df, cols):
    another = df.columns.difference(cols, sort=False).tolist()
    return df[cols + another]


# TODO: include scores predicted by machine (in addition to the two scorers), see model_selection.py for how to do this
def create_wide_df_from_GB_TD(hgs0=H_GS, fgs0=F_GS, hftd0=H_F_TD):

    hgs = deepcopy(hgs0)
    fgs = deepcopy(fgs0)
    hftd = deepcopy(hftd0)

    # gt hands
    hgs['train_type'] = "DoubleScoring"
    hgs['bodypart'] = hgs['bodypart_manual']
    hgs['laterality'] = hgs['laterality_manual']
    hgs['rater'] = 'true'

    hgs_ero = (hgs
               .filter(['train_type', 'id_date', 'bodypart', 'laterality', 'rater'] +
                       ero_pd_h + ['Base_MCIE'] + ero_wrist)
               )
    hgs_ero['score_type'] = 'ERO'

    hgs_jsn = (hgs
               .filter(['train_type', 'id_date', 'bodypart', 'laterality', 'rater'] +
                       jsn_finger + jsn_wrist)
               )
    hgs_jsn['score_type'] = 'JSN'

    # gt feet
    fgs['train_type'] = "DoubleScoring"
    fgs['bodypart'] = fgs['bodypart_manual']
    fgs['laterality'] = fgs['laterality_manual']
    fgs['rater'] = 'true'

    fgs_ero = (fgs
               .filter(['train_type', 'id_date', 'bodypart', 'laterality', 'rater'] +
                       ero_pd_f)
               )
    fgs_ero['score_type'] = 'ERO'

    fgs_jsn = (fgs
               .filter(['train_type', 'id_date', 'bodypart', 'laterality', 'rater'] +
                       jsn_feet)
               )
    fgs_jsn['score_type'] = 'JSN'

    r1 = (pd.concat([hgs_ero, hgs_jsn, fgs_ero, fgs_jsn], ignore_index=True, sort=False)
          .sort_values(['train_type', 'id_date', 'bodypart', 'laterality', 'score_type']))

    # double scoring
    hftd['train_type'] = "DoubleScoring"
    hftd['bodypart'] = hftd['bodypart_manual']
    hftd['laterality'] = hftd['laterality_manual']
    hftd['rater'] = 'preds'
    # hftd['score_type'] = ['ERO' if np.isnan(i) else 'JSN' for i in hftd['Rad_Carp']]

    # double scoring feet
    htd_ero = (hftd
               .query("bodypart == 'H'")
               .filter(['train_type', 'id_date', 'bodypart', 'laterality', 'rater'] +
                       ero_pd_h + ['Base_MCIE'] + ero_wrist)
               )
    htd_ero['score_type'] = 'ERO'

    htd_jsn = (hftd
               .query("bodypart == 'H'")
               .filter(['train_type', 'id_date', 'bodypart', 'laterality', 'rater'] +
                       jsn_finger + jsn_wrist)
               )
    htd_jsn['score_type'] = 'JSN'

    # double scoring feet
    ftd_ero = (hftd
               .query("bodypart == 'F'")
               .filter(['train_type', 'id_date', 'bodypart', 'laterality', 'rater'] +
                       ero_pd_f)
               )
    ftd_ero['score_type'] = 'ERO'

    ftd_jsn = (hftd
               .query("bodypart == 'F'")
               .filter(['train_type', 'id_date', 'bodypart', 'laterality', 'rater'] +
                       jsn_feet)
               )
    ftd_jsn['score_type'] = 'JSN'

    r2 = (pd.concat([htd_ero, htd_jsn, ftd_ero, ftd_jsn], ignore_index=True, sort=False)
          .sort_values(['train_type', 'id_date', 'bodypart', 'laterality', 'score_type', 'rater']))

    # fuse gt and double scoring
    r = (pd.concat([r1, r2], ignore_index=True, sort=False)
         .sort_values(['train_type', 'id_date', 'bodypart', 'laterality', 'score_type', 'rater'])
         .pipe(everything_after, ['train_type', 'id_date', 'bodypart', 'laterality', 'score_type', 'rater'])
         )

    # reshape to fit wide_df in model_selection.py
    r_long = r.melt(id_vars=["train_type", "id_date", "bodypart", "laterality", "score_type", "rater"],
                    var_name='joint',
                    value_name='score')

    r_semilong = r_long.pivot_table(index=["train_type", "id_date", "bodypart", "laterality", "score_type", "joint"],
                                    columns=['rater'],
                                    values=['score'],
                                    aggfunc='first'
                                    ).reset_index()

    r_semilong_flat = deepcopy(r_semilong)  # .groupby(['id_date'])[:, ('preds', 'ERO_finger')].sum()
    r_semilong_flat.columns = [re.sub("_*$", "", "__".join(n)).strip() for n in r_semilong_flat.columns.to_flat_index()]
    r_semilong_flat = r_semilong_flat.rename(columns={"score__true": "true", "score__preds": "preds"})
    r_semilong_flat = r_semilong_flat.astype({'preds': 'float64', 'true': 'float64'})

    r_wide = r_semilong_flat.pivot_table(index=["train_type", "id_date", "bodypart", "laterality", "score_type"],
                                         columns=['joint'],
                                         values=['true', 'preds'],
                                         aggfunc='first'
                                         ).reset_index()

    return deepcopy(r_wide)


wide_df = create_wide_df_from_GB_TD(hgs0=H_GS, fgs0=F_GS, hftd0=H_F_TD)

# calculate sum over distal/prox joint parts + summed scores over finger/wrist/foot of one extremity
for d in ['preds', 'true']:
    # sum over distal/prox joint part for fingers
    for sj in ero_sum_h:
        wide_df.loc[:, (d, sj)] = [min(p+d, 5) for p, d in zip(wide_df[d][sj+"P"], wide_df[d][sj+"D"])]
    # sum over distal/prox joint part for toes
    for sj in ero_sum_f:
        wide_df.loc[:, (d, sj)] = [min(p+d, 10) for p, d in zip(wide_df[d][sj+"P"], wide_df[d][sj+"D"])]
    # summed scores
    wide_df.loc[:, (d, 'ERO_finger')] = np.nansum(wide_df.loc[:, (d, ero_finger)].values, axis=1)
    wide_df.loc[:, (d, 'JSN_finger')] = np.nansum(wide_df.loc[:, (d, jsn_finger)].values, axis=1)
    if False:  # all([e in exclude for e in exclude_single_joints]):
        pass
        # wide_df.loc[:, (d, 'ERO_wrist')] = wide_df.loc[:, (d, 'ERO_wrist')]
        # wide_df.loc[:, (d, 'JSN_wrist')] = wide_df.loc[:, (d, 'JSN_wrist')]
    else:
        wide_df.loc[:, (d, 'ERO_wrist')] = np.nansum(wide_df.loc[:, (d, ero_wrist)].values, axis=1)
        wide_df.loc[:, (d, 'JSN_wrist')] = np.nansum(wide_df.loc[:, (d, jsn_wrist)].values, axis=1)
    wide_df.loc[:, (d, 'ERO_hand')] = wide_df.loc[:, (d, 'ERO_finger')] + wide_df.loc[:, (d, 'ERO_wrist')]
    wide_df.loc[:, (d, 'JSN_hand')] = wide_df.loc[:, (d, 'JSN_finger')] + wide_df.loc[:, (d, 'JSN_wrist')]
    wide_df.loc[:, (d, 'ERO_foot')] = np.nansum(wide_df.loc[:, (d, ero_feet)].values, axis=1)
    wide_df.loc[:, (d, 'JSN_foot')] = np.nansum(wide_df.loc[:, (d, jsn_feet)].values, axis=1)

# TODO: below, include the code that is applied to wide_df in model_selection.py to calculate the ICCs, etc.
# TODO: also produce confusion matrices for each joint group (finger, wrist, foot ERO/JSN)

# sum over extremities of same id_date

# # flatten multiindex df
flat_df = deepcopy(wide_df)  # .groupby(['id_date'])[:, ('preds', 'ERO_finger')].sum()
flat_df.columns = [re.sub("_*$", "", "__".join(n)).strip() for n in flat_df.columns.to_flat_index()]

# # sum over JSN/ERO (in ERO rows, JSN is nan or zero and vice versa, so summing them up is no problem)
tpp = ['true', 'preds']
sum_cols = ['ERO_finger', 'JSN_finger', 'ERO_wrist', 'JSN_wrist', 'ERO_hand', 'JSN_hand', 'ERO_foot', 'JSN_foot']

je_sum_df = flat_df.groupby(["train_type", "id_date", "bodypart", "laterality"])[
    [t + "__" + i
     for t in tpp
     for i in sum_cols]].sum().reset_index()

# # sum over hands and feet (in hand rows, feet scores are nan or zero and vice versa, so summing up is no problem)
hf_sum_df = je_sum_df.groupby(["train_type", "id_date"])[
    [t + "__" + i
     for t in tpp
     for i in sum_cols]].sum().reset_index()

# kappa whole id_date
idd_df = deepcopy(hf_sum_df)
for tp in tpp:
    idd_df.loc[:, tp + '__' + 'ERO_total'] = idd_df[[tp + '__' + i for i in ['ERO_hand', 'ERO_foot'
                                                                             ]]].sum(axis=1)
    idd_df.loc[:, tp + '__' + 'JSN_total'] = idd_df[[tp + '__' + i for i in ['JSN_hand', 'JSN_foot'
                                                                             ]]].sum(axis=1)
    idd_df.loc[:, tp + '__' + 'hand_total'] = idd_df[[tp + '__' + i for i in ['ERO_hand', 'JSN_hand'
                                                                              ]]].sum(axis=1)
    idd_df.loc[:, tp + '__' + 'foot_total'] = idd_df[[tp + '__' + i for i in ['ERO_foot', 'JSN_foot'
                                                                              ]]].sum(axis=1)
    idd_df.loc[:, tp + '__' + 'SvdH_total'] = idd_df[[tp + '__' + i for i in ['ERO_total', 'JSN_total'
                                                                              ]]].sum(axis=1)

nets = ["DoubleScoring"]
idd_cohen = {}.fromkeys(nets)
for n in nets:
    idd_cohen[n] = {}
    for t in ["ERO_finger", "JSN_finger", "ERO_wrist", "JSN_wrist", "ERO_hand", "JSN_hand", "ERO_foot", "JSN_foot",
              "ERO_total", "JSN_total", "hand_total", "foot_total",
              "SvdH_total"]:
        print(n, t)
        idd_cohen[n][t] = \
            cohen_kappa_score(idd_df.loc[(idd_df.train_type == n),
                                         'preds' + "__" + t].reset_index(drop=True),
                              idd_df.loc[(idd_df.train_type == n),
                                         'true' + "__" + t].reset_index(drop=True),
                              weights='quadratic')

idd_cohen_df = pd.DataFrame.from_dict(idd_cohen, orient='index')



In [ ]:
idd_cohen_df

In [ ]:

from rpy2.robjects import DataFrame, FloatVector, IntVector
from rpy2.robjects.packages import importr
r_lme4 = importr("lme4")     # should now import cleanly
r_icc = importr("ICC")
r_irr = importr("irr")
r_iccp = importr("psych")




# icc whole id_date
idd_icc = {}.fromkeys(nets)
for n in nets:
    idd_icc[n] = {}
    for t in ["ERO_finger", "JSN_finger", "ERO_wrist", "JSN_wrist", "ERO_hand", "JSN_hand",  "ERO_foot", "JSN_foot",
              "ERO_total", "JSN_total", "hand_total", "foot_total",
              "SvdH_total"]:
        print(n, t)
        df_nt = DataFrame({"preds": FloatVector(idd_df.loc[(idd_df.train_type == n),
                                                           'preds' + "__" + t].reset_index(drop=True)),
                           "true": FloatVector(idd_df.loc[(idd_df.train_type == n),
                                                          'true' + "__" + t].reset_index(drop=True))})

        # icc_nt = r_icc.ICCbare("preds", "true", data=df_nt)[0]

        icc_nt_dict = {r_iccp.ICC(df_nt)[0][0][k]: r_iccp.ICC(df_nt)[0][1][k]
                       for k in range(len(r_iccp.ICC(df_nt)[0][0]))}
        icc_nt = icc_nt_dict[icc_type]

        idd_icc[n][t] = icc_nt

idd_icc_df = pd.DataFrame.from_dict(idd_icc, orient='index')
display(idd_icc_df)

In [ ]:
icc_type = "ICC3"


## Version 1: Using R:
t = "JSN_foot"
v_label = idd_df[f"true__{t}"]
v_pred = idd_df[f"preds__{t}"]

df_nt = DataFrame({"preds": FloatVector(v_pred),
                   "true": FloatVector(v_label)})
icc_nt_dict = {r_iccp.ICC(df_nt)[0][0][k]: r_iccp.ICC(df_nt)[0][1][k]
                for k in range(len(r_iccp.ICC(df_nt)[0][0]))}

print("ICC via R::")
print("ICC3 = ", (icc_nt_dict["ICC3"]))


## Version 2: Using pingouin:
import pingouin as pg

n = len(v_pred)
df_long = pd.DataFrame({
    "target": np.repeat(np.arange(n), 2),
    "Raters": np.tile(["pred", "label"], n),
    "Rating": np.concatenate([v_pred, v_label]),
})
icc_tbl = pg.intraclass_corr(data=df_long, targets="target", raters="Raters", ratings="Rating")
print("ICC vai Pingouin::")
icc31 = icc_tbl.query("Type=='ICC3'")
display(icc31)


In [ ]:


# plot interrater reliability as heatmap

if display_agreement == "kappa":
    heatmap_df = idd_cohen_df
elif display_agreement == "icc":
    heatmap_df = idd_icc_df
else:
    raise ValueError("display_agreement must be either 'kappa' or 'icc'")

import seaborn as sns
sns.set()
ax = sns.heatmap(heatmap_df, annot=True, linewidths=.5, cmap="Blues", vmin=0.7, vmax=1)
plt.yticks(rotation=0)
ax.tick_params(length=0)
ax.xaxis.tick_top() # x axis on top
ax.xaxis.set_label_position('top')
plt.show()
#



In [ ]:
idd_icc_df

In [ ]:
idd_icc